## Imports

In [1]:
from ortools.sat.python import cp_model

## Parser

In [2]:
def parse_cap(filepath):
    with open(filepath) as f:
        tokens = f.read().split()
    
    idx = 0
    n_warehouses = int(tokens[idx]); idx += 1
    n_customers  = int(tokens[idx]); idx += 1

    capacity  = []
    fix_cost  = []
    for _ in range(n_warehouses):
        capacity.append(float(tokens[idx]));  idx += 1
        fix_cost.append(float(tokens[idx]));  idx += 1

    demand        = []
    transport     = []  # transport[i][j] = cost customer i → warehouse j
    for _ in range(n_customers):
        demand.append(float(tokens[idx])); idx += 1
        costs = []
        for _ in range(n_warehouses):
            costs.append(float(tokens[idx])); idx += 1
        transport.append(costs)

    return n_warehouses, n_customers, capacity, fix_cost, demand, transport

In [3]:
n_w, n_c, cap, fix, dem, trans = parse_cap("data/cap71.txt")

print(f"{n_w} warehouses, {n_c} customers")
print(f"Warehouse 0 capacity: {cap[0]}")
print(f"Warehouse 10 fixed cost: {fix[10]}")   # should be 0.0
print(f"Customer 0 demand: {dem[0]}")           # should be 146
print(f"Transport cost customer 0 → warehouse 0: {trans[0][0]}")  # should be 6739.725

16 warehouses, 50 customers
Warehouse 0 capacity: 58268.0
Warehouse 10 fixed cost: 0.0
Customer 0 demand: 146.0
Transport cost customer 0 → warehouse 0: 6739.725


## CP-SAT

In [4]:
def solve_cflp(n_w, n_c, cap, fix_cost, demand, transport):
    model = cp_model.CpModel()
    
    # --- Scaling (CP-SAT works with integers) ---
    SCALE = 100
    fix_cost_int   = [int(f * SCALE) for f in fix_cost]
    transport_int  = [[int(transport[i][j] * SCALE) for j in range(n_w)] for i in range(n_c)]
    demand_int     = [int(d) for d in demand]
    cap_int        = [int(c) for c in cap]

    # --- Decision variables ---
    # open_w[j] = 1 if warehouse j is opened
    open_w = [model.NewBoolVar(f'open_{j}') for j in range(n_w)]

    # assign[i][j] = 1 if customer i is served by warehouse j
    assign = [[model.NewBoolVar(f'assign_{i}_{j}') 
               for j in range(n_w)] for i in range(n_c)]

    # --- Constraints ---

    # 1. Each customer is served by exactly one warehouse
    for i in range(n_c):
        model.AddExactlyOne(assign[i][j] for j in range(n_w))

    # 2. A customer can only be assigned to an OPEN warehouse
    for i in range(n_c):
        for j in range(n_w):
            model.AddImplication(assign[i][j], open_w[j])

    # 3. Capacity: total demand assigned to j does not exceed its capacity
    for j in range(n_w):
        model.Add(
            sum(demand_int[i] * assign[i][j] for i in range(n_c)) <= cap_int[j] * open_w[j]
        )

    # --- Objective: minimize fixed cost + transport cost ---
    total_cost = (
        sum(fix_cost_int[j] * open_w[j] for j in range(n_w)) +
        sum(transport_int[i][j] * assign[i][j] for i in range(n_c) for j in range(n_w))
    )
    model.Minimize(total_cost)

    # --- Solve ---
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 60.0
    status = solver.Solve(model)

    # --- Display ---
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"Status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
        print(f"Total cost: {solver.ObjectiveValue() / SCALE:.2f}")
        print(f"Known optimum: 932615.75")
        print()
        
        open_warehouses = [j for j in range(n_w) if solver.Value(open_w[j])]
        print(f"Open warehouses ({len(open_warehouses)}): {open_warehouses}")
        
        print("\nCustomer assignment:")
        for i in range(n_c):
            for j in range(n_w):
                if solver.Value(assign[i][j]):
                    print(f"  Customer {i:2d} → Warehouse {j:2d}  "
                          f"(demand={demand[i]:.0f}, cost={transport[i][j]:.2f})")
    else:
        print("No solution found.")

    return solver, open_w, assign

In [5]:
n_w, n_c, cap, fix, dem, trans = parse_cap("data/cap71.txt")
solver, open_w, assign = solve_cflp(n_w, n_c, cap, fix, dem, trans)

Status: OPTIMAL
Total cost: 932615.64
Known optimum: 932615.75

Open warehouses (11): [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12]

Customer assignment:
  Customer  0 → Warehouse  7  (demand=146, cost=3847.10)
  Customer  1 → Warehouse 11  (demand=87, cost=1779.15)
  Customer  2 → Warehouse  0  (demand=672, cost=4914.00)
  Customer  3 → Warehouse  5  (demand=1337, cost=20071.71)
  Customer  4 → Warehouse  7  (demand=31, cost=955.58)
  Customer  5 → Warehouse  0  (demand=559, cost=6421.51)
  Customer  6 → Warehouse  1  (demand=2370, cost=28499.25)
  Customer  7 → Warehouse  2  (demand=1089, cost=6370.65)
  Customer  8 → Warehouse  7  (demand=33, cost=1211.92)
  Customer  9 → Warehouse  7  (demand=32, cost=546.40)
  Customer 10 → Warehouse  3  (demand=5495, cost=12638.50)
  Customer 11 → Warehouse 10  (demand=904, cost=2463.40)
  Customer 12 → Warehouse  5  (demand=1466, cost=5185.98)
  Customer 13 → Warehouse  0  (demand=143, cost=1953.74)
  Customer 14 → Warehouse  6  (demand=615, cost=7310.81

In [6]:
def solve_cflp_robust(n_w, n_c, cap, fix_cost, demand, transport, uncertainty=0.2):
    """
    uncertainty=0.2 means demand can be up to +20% higher.
    The solution must remain feasible in the worst case.
    """
    model = cp_model.CpModel()
    SCALE = 100

    fix_cost_int  = [int(f * SCALE) for f in fix_cost]
    transport_int = [[int(transport[i][j] * SCALE) for j in range(n_w)] for i in range(n_c)]
    
    # Worst-case demand = demand * (1 + uncertainty)
    demand_worst = [int(d * (1 + uncertainty)) for d in demand]
    cap_int      = [int(c) for c in cap]

    open_w = [model.NewBoolVar(f'open_{j}') for j in range(n_w)]
    assign = [[model.NewBoolVar(f'assign_{i}_{j}') for j in range(n_w)] for i in range(n_c)]

    # Same constraints as the base model
    for i in range(n_c):
        model.AddExactlyOne(assign[i][j] for j in range(n_w))

    for i in range(n_c):
        for j in range(n_w):
            model.AddImplication(assign[i][j], open_w[j])

    # Capacity constraint with worst-case demand
    for j in range(n_w):
        model.Add(
            sum(demand_worst[i] * assign[i][j] for i in range(n_c)) <= cap_int[j] * open_w[j]
        )

    total_cost = (
        sum(fix_cost_int[j] * open_w[j] for j in range(n_w)) +
        sum(transport_int[i][j] * assign[i][j] for i in range(n_c) for j in range(n_w))
    )
    model.Minimize(total_cost)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 60.0
    status = solver.Solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        open_warehouses = [j for j in range(n_w) if solver.Value(open_w[j])]
        print(f"[Robust +{int(uncertainty*100)}%] Cost: {solver.ObjectiveValue()/SCALE:.2f}")
        print(f"Open warehouses ({len(open_warehouses)}): {open_warehouses}")
    
    return solver

# Comparison for different uncertainty levels
for u in [0.0, 0.1, 0.2, 0.3]:
    solve_cflp_robust(n_w, n_c, cap, fix, dem, trans, uncertainty=u)

[Robust +0%] Cost: 932615.64
Open warehouses (11): [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12]
[Robust +10%] Cost: 932615.64
Open warehouses (11): [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12]
[Robust +20%] Cost: 932615.64
Open warehouses (11): [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12]
[Robust +30%] Cost: 932615.64
Open warehouses (11): [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12]


In [7]:
total_demand = sum(dem)
total_open_capacity = 11 * 58268

print(f"Nominal total demand : {total_demand:.0f}")
print(f"Demand +30%          : {total_demand * 1.3:.0f}")
print(f"Total open capacity  : {total_open_capacity:.0f}")

Nominal total demand : 58268
Demand +30%          : 75748
Total open capacity  : 640948


In [8]:
import random
import numpy as np

def generate_tight_instance(n_w=10, n_c=30, seed=42):
    """
    Synthetic instance where capacity is just sufficient
    → demand uncertainty forces different decisions
    """
    rng = random.Random(seed)
    
    demand   = [rng.randint(50, 500) for _ in range(n_c)]
    total_demand = sum(demand)
    
    # Capacity per warehouse = total demand / n_w * 1.2  (only 20% margin)
    capacity = [int(total_demand / n_w * 1.2)] * n_w
    fix_cost = [rng.uniform(1000, 5000) for _ in range(n_w)]
    
    transport = [[rng.uniform(100, 5000) for _ in range(n_w)] for _ in range(n_c)]
    
    return n_w, n_c, capacity, fix_cost, demand, transport

# Test
n_w, n_c, cap_s, fix_s, dem_s, trans_s = generate_tight_instance()

print("=== Tight synthetic instance ===")
for u in [0.0, 0.1, 0.2, 0.3]:
    solve_cflp_robust(n_w, n_c, cap_s, fix_s, dem_s, trans_s, uncertainty=u)

=== Tight synthetic instance ===
[Robust +0%] Cost: 42590.14
Open warehouses (9): [0, 1, 2, 3, 5, 6, 7, 8, 9]
[Robust +10%] Cost: 46388.13
Open warehouses (10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [9]:
total_demand = sum(dem_s)
total_capacity = sum(cap_s)

print(f"Nominal demand       : {total_demand}")
print(f"Demand +20%          : {total_demand * 1.2:.0f}")
print(f"Demand +30%          : {total_demand * 1.3:.0f}")
print(f"Total capacity (all) : {total_capacity}")

Nominal demand       : 7331
Demand +20%          : 8797
Demand +30%          : 9530
Total capacity (all) : 8790


In [10]:
def generate_tight_instance(n_w=10, n_c=30, seed=42, cap_margin=1.5):
    rng = random.Random(seed)
    demand       = [rng.randint(50, 500) for _ in range(n_c)]
    total_demand = sum(demand)
    # total capacity = cap_margin * nominal demand
    capacity  = [int(total_demand / n_w * cap_margin)] * n_w
    fix_cost  = [rng.uniform(1000, 5000) for _ in range(n_w)]
    transport = [[rng.uniform(100, 5000) for _ in range(n_w)] for _ in range(n_c)]
    return n_w, n_c, capacity, fix_cost, demand, transport

n_w, n_c, cap_s, fix_s, dem_s, trans_s = generate_tight_instance(cap_margin=1.5)

print("=== Synthetic instance (margin 1.5x) ===")
for u in [0.0, 0.1, 0.2, 0.3]:
    solve_cflp_robust(n_w, n_c, cap_s, fix_s, dem_s, trans_s, uncertainty=u)

=== Synthetic instance (margin 1.5x) ===
[Robust +0%] Cost: 38244.03
Open warehouses (7): [0, 2, 4, 5, 6, 7, 8]
[Robust +10%] Cost: 39503.41
Open warehouses (8): [1, 2, 4, 5, 6, 7, 8, 9]
[Robust +20%] Cost: 42108.78
Open warehouses (9): [0, 1, 2, 3, 5, 6, 7, 8, 9]
[Robust +30%] Cost: 44244.60
Open warehouses (9): [0, 1, 2, 3, 5, 6, 7, 8, 9]


In [11]:
total_demand = sum(dem_s)
total_capacity = sum(cap_s)

print(f"Nominal demand       : {total_demand}")
print(f"Demand +20%          : {total_demand * 1.2:.0f}")
print(f"Demand +30%          : {total_demand * 1.3:.0f}")
print(f"Total capacity (all) : {total_capacity}")

Nominal demand       : 7331
Demand +20%          : 8797
Demand +30%          : 9530
Total capacity (all) : 10990
